In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("argo_profiles_combined.csv", parse_dates=["time"])

# Sort by time & depth (important for interpolation)
df = df.sort_values(by=["time", "depth"]).reset_index(drop=True)

# Interpolate column-wise
df["temperature"] = df["temperature"].interpolate(method="linear", limit_direction="both")
df["salinity"] = df["salinity"].interpolate(method="linear", limit_direction="both")
df["oxygen"] = df["oxygen"].interpolate(method="linear", limit_direction="both")
df["chlorophyll"] = df["chlorophyll"].interpolate(method="linear", limit_direction="both")

# z-score filtering (remove points with |z| > 3)
from scipy.stats import zscore
for col in ["temperature","salinity","oxygen","chlorophyll"]:
    df = df[(np.abs(zscore(df[col], nan_policy="omit")) < 3)]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[["latitude","longitude","depth","temperature","oxygen","chlorophyll"]])
df_scaled = pd.DataFrame(scaled_features, columns=["latitude","longitude","depth","temperature","oxygen","chlorophyll"])
df_scaled["salinity"] = df["salinity"].values  # keep target unscaled

df_scaled.to_csv("argo_cleaned.csv", index=False)
print("Cleaned dataset saved.")


Cleaned dataset saved.


In [3]:
from sklearn.ensemble import IsolationForest

X = df[["latitude","longitude","depth","temperature","salinity","oxygen","chlorophyll"]].fillna(0)

iso = IsolationForest(contamination=0.05, random_state=42)  # 5% anomalies
df["anomaly"] = iso.fit_predict(X)

# Convert labels (-1 = anomaly, 1 = normal) → (1, 0)
df["anomaly"] = df["anomaly"].map({-1:1, 1:0})
print(df["anomaly"].value_counts())


anomaly
0    1721
1      91
Name: count, dtype: int64


In [4]:
df.to_csv("argo_labeled.csv", index=False)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

X = df[["latitude","longitude","depth","temperature","salinity","oxygen","chlorophyll"]]
y = df["anomaly"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Save
joblib.dump(clf, "anomaly_rf_model.pkl")

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       345
           1       1.00      0.56      0.71        18

    accuracy                           0.98       363
   macro avg       0.99      0.78      0.85       363
weighted avg       0.98      0.98      0.97       363

[[345   0]
 [  8  10]]


['anomaly_rf_model.pkl']

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import numpy as np

# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 2: Handle imbalance with SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)  # anomalies = 50% of normals
X_res, y_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_res.value_counts())

# Step 3: Train XGBoost with class weight
scale_pos_weight = (y_train.value_counts()[0] / y_train.value_counts()[1])

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    random_state=42
)

xgb.fit(X_res, y_res)

# Step 4: Predict probabilities
y_proba = xgb.predict_proba(X_test)[:,1]

# Adjust threshold (default=0.5)
threshold = 0.35   # lower → higher recall
y_pred = (y_proba >= threshold).astype(int)

# Step 5: Evaluate
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Before SMOTE: anomaly
0    1376
1      73
Name: count, dtype: int64
After SMOTE: anomaly
0    1376
1     688
Name: count, dtype: int64


c:\Users\priya\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:183: UserWarning: [16:04:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99       345
           1       0.78      1.00      0.88        18

    accuracy                           0.99       363
   macro avg       0.89      0.99      0.94       363
weighted avg       0.99      0.99      0.99       363

Confusion Matrix:
 [[340   5]
 [  0  18]]


In [7]:
import joblib
joblib.dump(xgb, "argo_anomaly_model.pkl")


['argo_anomaly_model.pkl']

In [12]:
import joblib
import pandas as pd

# Load trained model
xgb = joblib.load("argo_anomaly_model.pkl")

# Sample input (must match training features exactly)
sample = pd.DataFrame([{
    "latitude": -40.3,
    "longitude": 73.4,
    "depth": 980,
    "temperature": 3.8,
    "salinity": 34.5,
    "oxygen": 210,
    "chlorophyll": 0.4
}])

# Predict
pred = xgb.predict(sample)
prob = xgb.predict_proba(sample)

print("Prediction:", "Anomaly" if pred[0]==1 else "Normal")
print("Confidence:", prob)


Prediction: Anomaly
Confidence: [[0.04734677 0.9526532 ]]
